# NF-BoT-IoT-v3 — Phase 3: data preprocessing (cleaning)

**Owner:** Cyrine Chabbar (cleaning code prepared by Ahmed Hamouda with the same steps as the other datasets). Applies the
approved decisions of phase 2 for this dataset (`report/data_problems/phase2_data_problems.pdf`, section 4 and section 6).
Phase 2 evidence: `04_bot.ipynb`. The result has exactly the same 56 columns as the three other cleaned files.

| Step | What | Decision |
|---|---|---|
| 0 | Load the raw CSV (`L7_PROTO` as text: it has real decimal sub-codes here) | – |
| 1 | Drop the two `..._SECOND_BYTES` columns (essential: their NaN would make `clean()` delete 65% of Benign) | B2 |
| 2 | Split `L7_PROTO` into `L7_MAIN` + `L7_SUB` from the raw text | B3 |
| 3 | Remove exact duplicate rows (phase 2 found none) | shared rule |
| 4 | Check for flows that carry two labels (C5 rule; none expected) | C5 rule |
| 5 | One row per (behaviour, label), with a `count` column | Section 6 |
| 6 | Shared `clean()`: drop the IP columns, NaN / inf, duplicates | M3 + shared rule |
| 7 | `map_labels()` and drop `Label` | – |
| 8 | Checks and save | – |

**Not applied here, on purpose:** the CIC-2018 rule C7. Reconnaissance flows that look like Benign are kept (B7), and so are
identical DoS / DDoS floods (B6, same family). **Also kept:** Theft (B4, the only Data theft source), TTL = 0, TCP flows
without flags, oversized packets, outliers. **After the merge:** M1, M6, M7, M8, M9 (the 320 invalid DNS TTLs become 0 there).

16.9 million rows: about 6 GB of memory.

## Step 0 — Load the raw data

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.csv as pv
import pyarrow.parquet as pq

from src.config import *

pd.set_option("display.max_columns", 100)
SOURCE = "BoT"
path = RAW_DIR / DATASETS[SOURCE]["file"]

header = pd.read_csv(path, nrows=0).columns
# int32 halves the memory; pyarrow stops with an error if a value does not fit. DNS_TTL_ANSWER needs int64 (invalid values
# close to 2^32), the SECOND_BYTES columns are decimals with NaN / inf, and L7_PROTO is read as text (B3).
types = {c: pa.int32() for c in header if c not in ["IPV4_SRC_ADDR", "IPV4_DST_ADDR", "Attack"]}
for c in ["FLOW_START_MILLISECONDS", "FLOW_END_MILLISECONDS", "DNS_TTL_ANSWER"]:
    types[c] = pa.int64()
for c in ["SRC_TO_DST_SECOND_BYTES", "DST_TO_SRC_SECOND_BYTES"]:
    types[c] = pa.float64()
types["L7_PROTO"] = pa.string()
table = pv.read_csv(path, convert_options=pv.ConvertOptions(column_types=types))
# Convert one column at a time and free it from the arrow table (lower memory peak).
data = {}
for c in list(table.column_names):
    data[c] = table.column(c).to_pandas()
    table = table.drop_columns([c])
del table
df = pd.DataFrame(data, copy=False)
del data

raw_labels = df["Attack"].value_counts()
steps = []
keep = np.ones(len(df), dtype=bool)   # steps 3-4 mark removals here; step 5 keeps the marked rows in one go
def log(step, decision, what, rows=None):
    rows = len(df) if rows is None else int(rows)
    steps.append({"step": step, "decision": decision, "rows": rows, "columns": df.shape[1], "what": what})
    print(f"{step}: {rows:,} rows x {df.shape[1]} columns  ({what})")

def removed(mask, name):
    """Show what a removal takes away, per label (compared with the rows still kept before it)."""
    print(f"{name}: {int(mask.sum()):,} rows")
    if mask.any():
        attack = df["Attack"].to_numpy()
        before = pd.Series(attack[keep]).value_counts().rename("before")
        display(pd.Series(attack[mask]).value_counts().rename("removed").to_frame()
                .join(before).assign(after=lambda t: t["before"] - t["removed"]))

log("0 load", "-", "raw file")
raw_labels

0 load: 16,933,808 rows x 55 columns  (raw file)


Attack
DoS               8034190
DDoS              7150882
Reconnaissance    1695132
Benign              51989
Theft                1615
Name: count, dtype: int64

## Step 1 — Drop the two `..._SECOND_BYTES` columns (B2)
As in UNSW they hold NaN and infinity on every zero-duration flow, and the missing values hit the small classes: 64.9% of
Benign and 70.3% of Theft. `clean()` deletes every row with a NaN, so the columns must go first. Their information is in
the `..._AVG_THROUGHPUT` columns.

In [2]:
sb = ["SRC_TO_DST_SECOND_BYTES", "DST_TO_SRC_SECOND_BYTES"]
bad = (df[sb].isna() | np.isinf(df[sb])).any(axis=1)
print("rows that clean() would delete:", f"{int(bad.sum()):,}")
display((100 * bad.groupby(df["Attack"]).mean()).round(1).rename("% of the class"))
df = df.drop(columns=sb)
num = df.select_dtypes("number")
print("after: NaN cells", int(df.isna().sum().sum()), "| inf cells", int(np.isinf(num).sum().sum()))
del bad, num
log("1 drop SECOND_BYTES", "B2", "2 columns removed")

rows that clean() would delete: 1,052,105


Attack
Benign            64.9
DDoS               1.3
DoS                2.8
Reconnaissance    41.2
Theft             70.3
Name: % of the class, dtype: float64

after: NaN cells 0 | inf cells 0
1 drop SECOND_BYTES: 16,933,808 rows x 53 columns  (2 columns removed)


## Step 2 — Split `L7_PROTO` into `L7_MAIN` + `L7_SUB` (B3)
As in UNSW, `L7_PROTO` holds two nDPI codes in one decimal number (`10.16` = NetBIOS carrying SMBv1). Read as text, so the
second code is not lost. BoT-IoT writes every value as a decimal (`7.0` = HTTP with no sub-code), and a few values carry
floating-point noise from the export (`5.2120000000000015` is `5.212`): nDPI codes have at most 3 digits, so a longer
decimal part is rounded to 3 decimals.

In [3]:
def split_code(s):
    main, _, sub = s.partition(".")
    if len(sub) > 3:   # float noise in the raw text, e.g. 5.2120000000000015 = 5.212
        sub = f"{float(s):.3f}".partition(".")[2]
    return int(main), int(sub or 0)

code_map = {s: split_code(s) for s in df["L7_PROTO"].unique()}
print("raw values with float noise:", {s: code_map[s] for s in code_map if len(s.partition(".")[2]) > 3})
pos = df.columns.get_loc("L7_PROTO")
df.insert(pos, "L7_MAIN", df["L7_PROTO"].map({s: m for s, (m, _) in code_map.items()}).astype("int32"))
df.insert(pos + 1, "L7_SUB", df["L7_PROTO"].map({s: u for s, (_, u) in code_map.items()}).astype("int32"))
has_sub = (df["L7_SUB"] != 0).to_numpy()
one_digit = has_sub & (df["L7_SUB"] < 10).to_numpy()
print(f"flows with a sub-code: {has_sub.sum():,} | one-digit sub-codes (a lost trailing zero is possible, e.g. 7.5): {one_digit.sum():,}")
display(df.loc[has_sub, ["L7_PROTO", "L7_MAIN", "L7_SUB"]].value_counts().head(10))
df = df.drop(columns="L7_PROTO")
del has_sub, one_digit
log("2 split L7_PROTO", "B3", "L7_PROTO -> L7_MAIN + L7_SUB")

raw values with float noise: {'5.2120000000000015': (5, 212), '7.2120000000000015': (7, 212)}


flows with a sub-code: 1,851 | one-digit sub-codes (a lost trailing zero is possible, e.g. 7.5): 796


L7_PROTO            L7_MAIN  L7_SUB
7.5                 7        5         796
10.16               10       16        337
5.239               5        239       213
10.41               10       41        132
5.265               5        265       125
7.376               7        376       119
91.239              91       239        45
5.2120000000000015  5        212        27
5.283               5        283        26
91.283              91       283        13
Name: count, dtype: int64

2 split L7_PROTO: 16,933,808 rows x 54 columns  (L7_PROTO -> L7_MAIN + L7_SUB)

## Step 3 — Remove exact duplicate rows (shared rule)
Rows identical in every column, including IPs and the millisecond timestamp. Phase 2 found none in BoT-IoT; the step is kept
so every dataset goes through the same rules.

In [4]:
row_hash = pd.util.hash_pandas_object(df, index=False).to_numpy()
dup = pd.Series(row_hash).duplicated().to_numpy()
removed(dup, "exact duplicates")
keep &= ~dup
del row_hash
log("3 exact duplicates", "shared rule", f"{int(dup.sum()):,} copies removed", keep.sum())

exact duplicates: 0 rows
3 exact duplicates: 16,933,808 rows x 54 columns  (0 copies removed)


## Step 4 — Check for flows that carry two labels (C5 rule)
The same flow (identical in every column except the labels) with two different labels would be removed entirely, as in
CIC-2018. Phase 2 found none in BoT-IoT; this confirms it.

In [5]:
feat = [c for c in df.columns if c not in ("Label", "Attack")]
fh = pd.util.hash_pandas_object(df[feat], index=False).to_numpy()
n_labels = pd.Series(df["Attack"].to_numpy()[keep]).groupby(fh[keep]).nunique()
two = keep & np.isin(fh, n_labels.index[n_labels > 1])
removed(two, "flows with two labels")
keep &= ~two
del fh
log("4 two labels (check)", "C5 rule", f"{int(two.sum()):,} rows removed", keep.sum())

flows with two labels: 0 rows
4 two labels (check): 16,933,808 rows x 54 columns  (0 rows removed)


## Step 5 — One row per (behaviour, label), with `count` (section 6)
Repeats of the same behaviour with the same label become one row; `count` keeps how many flows it stands for. Runs before
the IPs are dropped, so different machines doing the same thing are counted as separate flows. BoT-IoT's floods repeat the
same packet millions of times, so this is where the file shrinks most; no attack flow is deleted.

In [6]:
flows_before = int(keep.sum())
df = collapse_behaviours(df, "Attack", keep=keep)   # only the rows kept by steps 3-4
del keep
print(f"{flows_before:,} flows -> {len(df):,} rows; sum of count = {df['count'].sum():,}")
assert df["count"].sum() == flows_before
display(pd.DataFrame({"raw flows": raw_labels, "rows kept": df["Attack"].value_counts(),
                      "largest count": df.groupby("Attack")["count"].max()}).fillna(0).astype(int).sort_values("raw flows", ascending=False))
log("5 collapse behaviours", "section 6", "repeats -> one row + count")

16,933,808 flows -> 1,563,315 rows; sum of count = 16,933,808


,raw flows,rows kept,largest count
Attack,,,
DoS,8034190,354734,122631
DDoS,7150882,287453,86005
Reconnaissance,1695132,898729,22084
Benign,51989,21951,1536
Theft,1615,448,1090


5 collapse behaviours: 1,563,315 rows x 55 columns  (repeats -> one row + count)


## Step 6 — Shared `clean()` (`src/config.py`)

In [7]:
rows_before = len(df)
df, stats = clean(df, SOURCE)
print("rows lost in clean():", rows_before - len(df))
log("6 clean()", "M3 + shared rule", f"IP columns dropped; {rows_before - len(df)} rows lost")
stats

rows lost in clean(): 0
6 clean(): 1,563,315 rows x 53 columns  (IP columns dropped; 0 rows lost)


{'source_dataset': 'BoT',
 'rows_raw': 1563315,
 'cols_raw': 55,
 'id_columns_dropped': 'IPV4_SRC_ADDR, IPV4_DST_ADDR',
 'inf_values': 0,
 'nan_values': 0,
 'imputed_columns': 'none',
 'rows_with_nan': 0,
 'duplicate_rows': 0,
 'rows_clean': 1563315,
 'cols_clean': 53}

## Step 7 — Shared labels (`map_labels`) and drop `Label`
BoT-IoT's labels are already clean (Benign, DDoS, DoS, Reconnaissance, Theft); they get their families from the taxonomy.

In [8]:
df = map_labels(df, "Attack", SOURCE)
assert (df["is_attack"].astype(int) == df["Label"].astype(int)).all(), "Label and is_attack disagree"
df = df.drop(columns="Label")
display(df.groupby(["attack_family", "attack_label"], observed=True)["count"].agg(rows="size", flows="sum"))
log("7 map_labels + drop Label", "-", "4 label columns added, Label dropped")

rows    flows
attack_family     attack_label                   
Benign            Benign           21951    51989
Data theft        Theft              448     1615
Denial of service DDoS            287453  7150882
                  DoS             354734  8034190
Reconnaissance    Reconnaissance  898729  1695132

7 map_labels + drop Label: 1,563,315 rows x 56 columns  (4 label columns added, Label dropped)


## Step 8 — Checks and save

In [9]:
checks = validate(df)
print("validate(): total violations", int(checks["violations"].sum()))
num = df.select_dtypes("number")
print("NaN cells:", int(df.isna().sum().sum()), "| inf cells:", int(np.isinf(num).sum().sum()))
print("IP columns left:", [c for c in df.columns if "ADDR" in c])
for other in ["UNSW", "CIC18", "ToN"]:
    f = PROCESSED_DIR / f"{other}_clean.parquet"
    print(f"same columns and order as {other}_clean:", pq.read_schema(f).names == list(df.columns) if f.exists() else "file not found")
print(f"final: {len(df):,} rows x {df.shape[1]} columns | flows represented {int(df['count'].sum()):,} of {int(raw_labels.sum()):,} raw")

validate(): total violations 0


NaN cells: 0 | inf cells: 0
IP columns left: []
same columns and order as UNSW_clean: True
same columns and order as CIC18_clean: True
same columns and order as ToN_clean: True
final: 1,563,315 rows x 56 columns | flows represented 16,933,808 of 16,933,808 raw


In [10]:
log("8 final", "-", "cleaned table")
lab_map = map_labels(pd.DataFrame({"Attack": raw_labels.index}), "Attack", SOURCE).set_index("Attack")["attack_label"]
labels = cleaning_report(steps, raw_labels.groupby(lab_map).sum(), df, SOURCE)
out = save_clean(df, SOURCE)
print("saved", out.relative_to(ROOT), f"({out.stat().st_size / 1e6:.0f} MB)")
display(pd.DataFrame(steps))
labels.sort_values("flows_raw", ascending=False)

8 final: 1,563,315 rows x 56 columns  (cleaned table)


saved data\processed\BoT_clean.parquet (24 MB)


,step,decision,rows,columns,what
0,0 load,-,16933808,55,raw file
1,1 drop SECOND_BYTES,B2,16933808,53,2 columns removed
2,2 split L7_PROTO,B3,16933808,54,L7_PROTO -> L7_MAIN + L7_SUB
3,3 exact duplicates,shared rule,16933808,54,0 copies removed
4,4 two labels (check),C5 rule,16933808,54,0 rows removed
5,5 collapse behaviours,section 6,1563315,55,repeats -> one row + count
6,6 clean(),M3 + shared rule,1563315,53,IP columns dropped; 0 rows lost
7,7 map_labels + drop Label,-,1563315,56,"4 label columns added, Label dropped"
8,8 final,-,1563315,56,cleaned table


,source_dataset,flows_raw,rows_clean,flows_represented,attack_family
attack_label,,,,,
DoS,BoT,8034190,354734,8034190,Denial of service
DDoS,BoT,7150882,287453,7150882,Denial of service
Reconnaissance,BoT,1695132,898729,1695132,Reconnaissance
Benign,BoT,51989,21951,51989,Benign
Theft,BoT,1615,448,1615,Data theft
